In [43]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from datasets import load_dataset
from scipy import sparse
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

In [44]:
print("Loading IMDb dataset...")
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split

dataset = load_dataset("imdb")

train_size = 25000   # giảm để chạy nhanh
test_size  = 3000

train_data = dataset['train'].shuffle(seed=42).select(range(train_size))
test_data  = dataset['test'].shuffle(seed=42).select(range(test_size))

train_df = pd.DataFrame({
    'full_text': train_data['text'],
    'label': train_data['label']   # 0 = negative, 1 = positive
})

test_df = pd.DataFrame({
    'full_text': test_data['text'],
    'label': test_data['label']
})

print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")

print("\nLabel distribution:")
print(train_df['label'].value_counts().sort_index())

# Split validation giống AG News
train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df['label'],
    random_state=42
)

Loading IMDb dataset...
Training samples: 25000
Test samples: 3000

Label distribution:
label
0    12500
1    12500
Name: count, dtype: int64


In [45]:
import numpy as np

class KNN_Optimized:
    def __init__(self, k=5):
        self.k = k

    def fit(self, X, y):
        self.X_train = X.toarray() if hasattr(X, "toarray") else X
        self.y_train = y
        self.classes_ = np.unique(y)

        self.X_train_norm = self._normalize(self.X_train)

    def _normalize(self, X):
        norm = np.linalg.norm(X, axis=1, keepdims=True)
        norm[norm == 0] = 1e-9
        return X / norm

    def _cosine_similarity(self, X):
        X = X.toarray() if hasattr(X, "toarray") else X
        X_norm = self._normalize(X)

        return np.dot(X_norm, self.X_train_norm.T)

    def predict(self, X):
        sims = self._cosine_similarity(X)

        knn_idx = np.argsort(sims, axis=1)[:, -self.k:]
        knn_labels = self.y_train[knn_idx]

        preds = []
        for labels in knn_labels:
            values, counts = np.unique(labels, return_counts=True)
            preds.append(values[np.argmax(counts)])

        return np.array(preds)

    def predict_proba(self, X):
        sims = self._cosine_similarity(X)
        knn_idx = np.argsort(sims, axis=1)[:, -self.k:]
        knn_labels = self.y_train[knn_idx]

        proba = np.zeros((X.shape[0], len(self.classes_)))

        for i, labels in enumerate(knn_labels):
            for j, c in enumerate(self.classes_):
                proba[i, j] = np.sum(labels == c)

        proba /= self.k
        return proba


In [46]:
import numpy as np

class KNN_WeightedCosine:
    def __init__(self, k=5):
        self.k = k

    def fit(self, X, y):
        self.X_train = X.toarray() if hasattr(X, "toarray") else X
        self.y_train = np.asarray(y)
        self.classes_ = np.unique(self.y_train)

        self.X_train_norm = self._normalize(self.X_train)

    def _normalize(self, X):
        norm = np.linalg.norm(X, axis=1, keepdims=True)
        norm[norm == 0] = 1e-9
        return X / norm

    def _cosine_similarity(self, X):
        X = X.toarray() if hasattr(X, "toarray") else X
        X_norm = self._normalize(X)
        return np.dot(X_norm, self.X_train_norm.T)

    def predict(self, X):
        sims = self._cosine_similarity(X)

        knn_idx = np.argpartition(sims, -self.k, axis=1)[:, -self.k:]

        preds = []
        for i, idx in enumerate(knn_idx):
            labels = self.y_train[idx]
            weights = sims[i, idx]

            weights = np.clip(weights, 0, None)

            class_scores = {}
            for c in self.classes_:
                class_scores[c] = weights[labels == c].sum()

            preds.append(max(class_scores, key=class_scores.get))

        return np.array(preds)

    def predict_proba(self, X):
        sims = self._cosine_similarity(X)
        knn_idx = np.argpartition(sims, -self.k, axis=1)[:, -self.k:]

        proba = np.zeros((X.shape[0], len(self.classes_)))

        for i, idx in enumerate(knn_idx):
            labels = self.y_train[idx]
            weights = np.clip(sims[i, idx], 0, None)

            for j, c in enumerate(self.classes_):
                proba[i, j] = weights[labels == c].sum()

            total = proba[i].sum()
            if total > 0:
                proba[i] /= total
            else:
                proba[i] = 1.0 / len(self.classes_)

        return proba


In [47]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words='english'
)

X_train = tfidf.fit_transform(train_df['full_text'])
X_test  = tfidf.transform(test_df['full_text'])

y_train = train_df['label'].values
y_test  = test_df['label'].values

In [48]:
knn = KNN_Optimized(k=5)
knn.fit(X_train, y_train)
knn_probs_test = knn.predict_proba(X_test)
acc_knn = accuracy_score(y_test, knn.predict(X_test))
print(f"KNN: {acc_knn:.4f}")

KNN: 0.6517


In [49]:
knn1 = KNN_WeightedCosine(k=5)
knn1.fit(X_train, y_train)
knn1_probs_test = knn1.predict_proba(X_test)
acc_knn1 = accuracy_score(y_test, knn1.predict(X_test))
print(f"KNN: {acc_knn1:.4f}")

KNN: 0.6523
